In [2]:
import nltk
import spacy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter, defaultdict
from tqdm import tqdm
from nltk import (
  CFG,  # Context-Free Grammar 
  PCFG  # Probabilistic Context-Free Grammar
)
from nltk.parse import (
  ChartParser,    # Chart Parsing Algorithm
  ViterbiParser   # Viterbi Parsing Algorithm
)
from nltk.tokenize import (
  word_tokenize,  # Tokenize a sentence into words
  sent_tokenize,  # Tokenize text into sentences
)
from spacy import displacy

try:
  nltk.data.find('tokenizers/punkt')
except LookupError:
  nltk.download('punkt')

nlp = spacy.load("es_core_news_sm")

In [3]:
sentences = [
  "Como fan de las series españolas y de Najwa, esto duele, la serie es muy mala",
  "Manu Ríos da para lo que da, enseñar cacho, Najwa hace de mala, papel repetido que no aporta ninguna capa nueva",
  "Telenovela de mediodía con un guión mediocre y diálogos planos",
  "En aspectos técnicos como fotografía, sonido, también deja que desear",
  "Lo peor de Carlos Montero, de largo."
]

# Tokenización

In [4]:
tokens_by_sent = {}
for i,sent in enumerate(sentences):
  clean_sent = sent.lower()
  tokens_by_sent[i] = word_tokenize(clean_sent, language='spanish')

for idx in tokens_by_sent.keys():
  print(f"Doc: {idx+1}: {sentences[idx]}")
  print(f"Tokens: {tokens_by_sent[idx]}")

Doc: 1: Como fan de las series españolas y de Najwa, esto duele, la serie es muy mala
Tokens: ['como', 'fan', 'de', 'las', 'series', 'españolas', 'y', 'de', 'najwa', ',', 'esto', 'duele', ',', 'la', 'serie', 'es', 'muy', 'mala']
Doc: 2: Manu Ríos da para lo que da, enseñar cacho, Najwa hace de mala, papel repetido que no aporta ninguna capa nueva
Tokens: ['manu', 'ríos', 'da', 'para', 'lo', 'que', 'da', ',', 'enseñar', 'cacho', ',', 'najwa', 'hace', 'de', 'mala', ',', 'papel', 'repetido', 'que', 'no', 'aporta', 'ninguna', 'capa', 'nueva']
Doc: 3: Telenovela de mediodía con un guión mediocre y diálogos planos
Tokens: ['telenovela', 'de', 'mediodía', 'con', 'un', 'guión', 'mediocre', 'y', 'diálogos', 'planos']
Doc: 4: En aspectos técnicos como fotografía, sonido, también deja que desear
Tokens: ['en', 'aspectos', 'técnicos', 'como', 'fotografía', ',', 'sonido', ',', 'también', 'deja', 'que', 'desear']
Doc: 5: Lo peor de Carlos Montero, de largo.
Tokens: ['lo', 'peor', 'de', 'carlos', 'mo

# Procesar Textos con Spacy

In [5]:
def process_spacy(texts, batch_size=64):
  return nlp.pipe(texts, batch_size=batch_size)
docs = [doc for doc in tqdm(process_spacy(sentences, batch_size=64), total=len(sentences))]

100%|██████████| 5/5 [00:00<00:00, 166.67it/s]


**Leyenda**:
- `NOUN`: Sustantivo
- `VERB`: Verbo
- `ADJ`: Adjetivo
- `ADV`: Adverbio
- `PROPN`: Nombre Propio
- `DET`: Determinante
- `PRON`: Pronombre
- `ADP`: Preposición
- `CCONJ`: Conjunción
- `SCONJ`: Conjunción Sub
- `INTJ`: Interjección
- `NUM`: Número

In [14]:
categories = defaultdict(set)

tags = ["NOUN", "VERB", "AUX", "ADJ", "ADV", "PROPN", "DET", "PRON", "ADP", "CCONJ", "SCONJ", "INTJ", "NUM", "PUNCT"]
# Puede extenderse con: `NOUN__Gender=Masc|Number=Sing`, `NOUN__Gender=Fem|Number=Sing`, `VERB__Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin` 

for doc in docs:
  for token in doc:
    is_found = False
    for tag in tags:
      if token.pos_ == tag:
        categories[tag].add(token.text.lower())
        is_found = True
    if not is_found:
      print(f"Dont Found: {token.text} | {token.pos_}")
      # categories["OTHER"].add(token.text.lower())

for idx in categories.keys():
  print(f"TAG({idx}) = {categories[idx]}")

TAG(SCONJ) = {'que', 'como'}
TAG(NOUN) = {'mediodía', 'mala', 'aspectos', 'fotografía', 'capa', 'guión', 'sonido', 'largo', 'cacho', 'serie', 'series', 'diálogos', 'repetido', 'fan', 'papel'}
TAG(ADP) = {'con', 'para', 'en', 'de'}
TAG(DET) = {'ninguna', 'la', 'las', 'un'}
TAG(ADJ) = {'mediocre', 'españolas', 'mala', 'técnicos', 'peor', 'nueva', 'planos'}
TAG(CCONJ) = {'y'}
TAG(PROPN) = {'manu', 'ríos', 'montero', 'telenovela', 'najwa', 'carlos'}
TAG(PUNCT) = {',', '.'}
TAG(PRON) = {'esto', 'que', 'lo'}
TAG(VERB) = {'aporta', 'hace', 'duele', 'da', 'desear', 'deja', 'enseñar'}
TAG(AUX) = {'es'}
TAG(ADV) = {'también', 'no', 'muy'}


# Construcción de Reglas CFG

In [19]:
def build_cfg(categories):
  lexical_rules = []
  for tag, items in categories.items():
    if items:
      alts = " | ".join(sorted({f"'{w}'" for w in items}))
      lexical_rules.append(f"{tag} -> {alts}")
  structural_rules = [
    "S -> CLAUSE",
    "S -> CLAUSE PUNCT",
    "S -> CLAUSE PUNCT S",
    "S -> S PUNCT",
    "S -> S PUNCT CLAUSE",
    "S -> S CCONJ CLAUSE",
    "S -> S SCONJ CLAUSE",
    "S -> PP CLAUSE",
    "S -> SCONJ NP PUNCT S",
    "CLAUSE -> NP VP",
    "CLAUSE -> VP NP",
    "CLAUSE -> VP",
    "CLAUSE -> NP",
    "CLAUSE -> ADVP VP",
    "CLAUSE -> NP SCONJ VP",
    "CLAUSE -> NP PRON VP",
    "CLAUSE -> SCONJ NP",
    "NP -> DET NOUN",
    "NP -> DET PROPN",
    "NP -> NOUN",
    "NP -> PROPN",
    "NP -> DET NOUN ADJP",
    "NP -> DET NOUN PP",
    "NP -> NOUN ADJP",
    "NP -> NOUN PP",
    "NP -> NP PP",
    "NP -> NP CCONJ NP",
    "NP -> PRON",
    "NP -> PRON ADJP",
    "NP -> DET ADJP",
    "ADJP -> ADJ",
    "ADJP -> ADV ADJ",
    "VP -> VERB",
    "VP -> VERB NP",
    "VP -> VERB ADJP",
    "VP -> VERB PP",
    "VP -> VERB ADVP",
    "VP -> ADVP VERB",
    "VP -> VERB SCONJ VERB",
    "VP -> AUX",
    "VP -> AUX NP",
    "VP -> AUX ADJP",
    "VP -> AUX PP",
    "VP -> AUX ADVP",
    "ADVP -> ADV",
    "PP -> ADP NP",
    "PP -> ADP ADJP",
    "PP -> PP CCONJ PP"
  ]
  return "\n".join(structural_rules + lexical_rules)

grammar_text = build_cfg(categories)
grammar = CFG.fromstring(grammar_text)
print(grammar)

Grammar with 84 productions (start state = S)
    S -> CLAUSE
    S -> S PUNCT CLAUSE
    S -> S CCONJ CLAUSE
    S -> S SCONJ CLAUSE
    CLAUSE -> NP VP
    CLAUSE -> VP NP
    CLAUSE -> VP
    CLAUSE -> NP
    NP -> DET NOUN
    NP -> DET PROPN
    NP -> NOUN
    NP -> PROPN
    NP -> DET NOUN ADJP
    NP -> DET NOUN PP
    NP -> PRON
    ADJP -> ADJ
    ADJP -> ADV ADJ
    VP -> VERB
    VP -> VERB NP
    VP -> VERB ADJP
    VP -> VERB PP
    VP -> VERB ADVP
    VP -> AUX
    VP -> AUX NP
    VP -> AUX ADJP
    VP -> AUX PP
    VP -> AUX ADVP
    ADVP -> ADV
    PP -> ADP NP
    SCONJ -> 'como'
    SCONJ -> 'que'
    NOUN -> 'aspectos'
    NOUN -> 'cacho'
    NOUN -> 'capa'
    NOUN -> 'diálogos'
    NOUN -> 'fan'
    NOUN -> 'fotografía'
    NOUN -> 'guión'
    NOUN -> 'largo'
    NOUN -> 'mala'
    NOUN -> 'mediodía'
    NOUN -> 'papel'
    NOUN -> 'repetido'
    NOUN -> 'serie'
    NOUN -> 'series'
    NOUN -> 'sonido'
    ADP -> 'con'
    ADP -> 'de'
    ADP -> 'en'
    ADP -> '

In [20]:

parser = ChartParser(grammar)
parsed = []
for i in range(len(sentences)):
  toks = tokens_by_sent[i]
  trees = list(parser.parse(toks))
  print(f"Sentence {i+1} parses: {len(trees)}")
  if trees:
    print(trees[0])
  parsed.append(trees)

Sentence 1 parses: 0
Sentence 2 parses: 0
Sentence 3 parses: 0
Sentence 4 parses: 0
Sentence 5 parses: 0
